# Gradient evaluation in ParaQeet

ParaQeet is desgined in a top-down modular structure, where each module can be differentiated. Modules providing gradient evaluation derive from the base class $\texttt{Differentiable}$. All the classes deriving from $\texttt{Differentiable}$ provide a `get_value_and_gradient` method that returns the gradient. It build on the idea of the *semi-AD* [Georz2022] and combines AD with analytic gradients from quantum optimal control to provide a resource efficient and flexible framework.

This notebook provides an overview of the gradient computation in all the modules and aims to explain how gradients are chained across the package.

## Chain rule based gradient computation

The aim here is to compute the gradients of an objective function $(\mathcal{F})$, a $\texttt{Measurement}$ in ParaQeet, with respect to the parameters of the input pulse $(\boldsymbol{\alpha})$.
Computing this derivative for a state-transfer fidelity $(\mathcal{F}(t) = |\phi|^2 =  |\langle \lambda | \psi(t) \rangle|^2)$ for a pulse $(\Omega(t))$ would lead to computing the terms
$$\frac{\partial \mathcal{F}(t) }{\partial \alpha} = \frac{\partial \mathcal{F}(t)}{\partial \phi}  \frac{\partial \phi}{\partial \Omega} \frac{\partial \Omega}{\partial \alpha}.$$

Practically the pulse contains multiple elements, and objective function may be a sum of multiple fidelities. Computing the derivative, thus, amounts to systematic use of chain rule. 
This is computed by forward propagating the gradients through the modules
$$\frac{\partial v_k }{\partial \alpha} = \sum_{i < k} \frac{\partial v_k} {\partial v_i} \frac{\partial v_i}{\partial \alpha},$$
where $v_k$ are values of the intermediate steps, and $v_N = \mathcal{F}(t)$ for some $N$.

Thus each module computes the gradients for its own parameters, and also collects the gradients propagated through the prior modules.

### 1. The $\texttt{signal}$ module
Here signal generation is modelled by modeling the components used in classical signal generation, including $\texttt{Waveform}$ generator, $\texttt{LocalOscillator}$, $\texttt{IQMixer}$, etc. 

The $\texttt{Waveform}$ class derives from the $\texttt{Optimizable}$ and $\texttt{Differentiable}$ base classes and offers the `get_value` and `get_value_and_gradient` methods. This class implements a fallback to automatic differentiation (AD) for computing gradients incase a `get_value_and_gradient` method is not specified. It expects the `_evaluate` method to be a pure-JAX function, with arguments as `jax.Array`. Further, all the classes deriving from $\texttt{Waveform}$ support differentiation of complex output.


The $\texttt{signal}$ module includes the $\texttt{Envelopes}$ class which defines the shape of a pulse. Lets define an example envelope of a Gaussian $$\Omega(t) = A \, e^{-\frac{(t - \mu)^2}{2\sigma^2}}.$$ And try to plot its gradients.

In [ ]:
from functools import partial

import jax.numpy as jnp
import numpy as np
from jax import jit

from paraqeet import Array, Quantity
from paraqeet.signal.envelopes import Envelope


class ExampleEnvelope(Envelope):
    _amplitude: Quantity
    _t_final: Quantity
    _t_up: Quantity
    _t_down: Quantity
    _ramp_time: Quantity

    def __init__(self):
        self._amplitude = Quantity(
            20e6,
            min_value=jnp.array(0.0),
            max_value=jnp.array(50e6),
            name="amplitude",
        )

        self._mu = Quantity(
            12e-9,
            min_value=jnp.array(0.0),
            max_value=jnp.array(20e-9),
            name="mu",
        )

        self._sigma = Quantity(
            5e-9,
            min_value=jnp.array(0.0),
            max_value=jnp.array(10e-9),
            name="sigma",
        )

    def get_parameters(self):
        return [self._amplitude, self._mu, self._sigma]

    @partial(jit, static_argnums=(0,))
    def _evaluate(self, amp: Array, mu: Array, sigma: Array, t: Array):
        return jnp.squeeze(amp * jnp.exp(-((t - mu) ** 2 / (2 * sigma**2))))

    def get_value(self, times: Array | float) -> Array:
        amp = self._amplitude.get_value()
        mu = self._mu.get_value()
        sigma = self._sigma.get_value()
        return self._evaluate(amp, mu, sigma, times)


tone = ExampleEnvelope()

Plotting the signal

In [ ]:
import matplotlib.pyplot as plt
from plotting import plot_signal

t_final = 20e-9
ts = np.linspace(0, t_final, 501)
sig, grads = tone.get_value_and_gradient(ts)

fig, ax = plt.subplots(1, figsize=(5, 3))
plot_signal(tone, ts, ax, linestyle="--")
plt.show()

Computing the gradients using the `get_value_and_gradient` method, we get 

In [ ]:
value, grads = tone.get_value_and_gradient(ts)
grads

The gradients are currently empty as the `OptimizationMap` is not defined yet. Lets define the optmap and add the `tone` to it.

In [ ]:
from paraqeet.optimization_map import OptimizationMap

optmap = OptimizationMap()
optmap.add(tone)
optmap.register_params_with_optimizables()
optmap

In [ ]:
value, grads = tone.get_value_and_gradient(ts)
grads.shape

Note that the shape of the gradients is (t, num_params). Here there are 3 parameters added to the `optmap`.

Lets compare the gradients with analytical formulas

In [ ]:
def der_gaussian(amp, mu, sigma, t):
    # Derivative w.r.t amp
    f = jnp.exp(-((t - mu) ** 2) / (2 * sigma**2))
    # Derivative w.r.t mu
    dmu = amp * f * (t - mu) / sigma**2
    # Derivative w.r.t sigma
    dsigma = amp * f * ((t - mu) ** 2 / sigma**3)
    return f, dmu, dsigma


params = tone.get_parameters()
damp, dmu, dsigma = der_gaussian(params[0].get_value(), params[1].get_value(), params[2].get_value(), ts)

fig, ax = plt.subplots(3, figsize=(5, 5), sharex=True)
ax[0].plot(ts / 1e-9, grads[:, 0, ...], linestyle="-", label="AD")
ax[0].plot(ts / 1e-9, damp, linestyle="--", label="Analytical")
ax[0].grid(True, linestyle=(1, (1, 5)), linewidth=1)
ax[0].legend()

ax[1].plot(ts / 1e-9, grads[:, 1, ...], linestyle="-", label="AD")
ax[1].plot(ts / 1e-9, dmu, linestyle="--", label="Analytical")
ax[1].grid(True, linestyle=(1, (1, 5)), linewidth=1)
ax[1].legend()

ax[2].plot(ts / 1e-9, grads[:, 2, ...], linestyle="-", label="AD")
ax[2].plot(ts / 1e-9, dsigma, linestyle="--", label="Analytical")
ax[2].grid(True, linestyle=(1, (1, 5)), linewidth=1)
ax[2].legend()

ax[0].set_ylabel(r"$\frac{\partial \Omega(t)}{\partial A}$")
ax[1].set_ylabel(r"$\frac{\partial \Omega(t)}{\partial \mu}$")
ax[2].set_ylabel(r"$\frac{\partial \Omega(t)}{\partial \sigma}$")
ax[-1].set_xlabel("Time [ns]")
plt.show()

Further, the signal module includes filter functions and $\texttt{DRAGMixer}$ to modify the pulse shape. Lets include a $\texttt{FlatTopGaussianFilter}$ $(\epsilon(t) = s(t) \Omega(t))$ on top of this pulse and add the DRAG component.
The final pulse would be $$ \mathcal{E}(t) = s(t) \Omega(t) - i \frac{\dot{\epsilon(t)}}{\delta}.$$

The derivative of this pulse wrt parameters like $\mu$ can be evaluated by
$$ \frac{\partial \mathcal{E}(t)}{\partial \mu} = s(t) \frac{\partial \Omega(t)}{\partial \mu} - \frac{i}{\delta} \frac{\partial^2  (s(t) \Omega(t))}{\partial \mu \, \partial t} $$



In [ ]:
from paraqeet.signal.waveform import DRAGMixer, FlatTopGaussianFilter

drag_tone = DRAGMixer(tone)
filtered_tone = FlatTopGaussianFilter(
    envelopes=drag_tone, t_final=Quantity(t_final, 1 / 2 * t_final, 2 * t_final, name="t_final")
)

optmap.remove(tone)
optmap.add(drag_tone)
optmap.register_params_with_optimizables()
optmap

In [ ]:
fig, ax = plt.subplots(1, figsize=(5, 3))

plot_signal(drag_tone, ts, ax, linestyle="--")
plt.show()

Lets plot the derivative with $\mu$

In [ ]:
value, grads = drag_tone.get_value_and_gradient(ts)

fig, ax = plt.subplots(1, figsize=(5, 3), sharex=True)
ax.plot(ts / 1e-9, np.real(grads[:, 1, ...]), linestyle="--", label="Real")
ax.plot(ts / 1e-9, np.imag(grads[:, 1, ...]), linestyle="--", label="Imag")
ax.grid(True, linestyle=(1, (1, 5)), linewidth=1)
ax.legend()
ax.set_xlabel("Time [ns]")
plt.show()

Now there is the added imaginary component in the derivative.

Finally, the `Generator` class gathers the gradients from individual components and returns them.

In [ ]:
from paraqeet.signal.iq_mixer import IQMixer

gen = IQMixer(envelopes=[tone])

optmap.remove(drag_tone)
optmap.add(gen)
optmap.register_params_with_optimizables()
optmap

In [ ]:
value, grads = gen.get_value_and_gradient(ts)
grads.shape

Note that now that the optmap has 5 elements, the gradients generated by the generator is also 5.

### 2. The $\texttt{Model}$ module

$\texttt{Model}$ includes the Hamiltonian, Drive and coupling. Due to the parameter dependence are only in the coefficient of the operators, computing derivatives just results in multiply the signal gradient with the respective operator. For a Hamiltonian of the from $H(t) = H_0 + u(t) H_d$, where $u(t)$ is the drive signal,
$$ \frac{\partial H(t)}{\partial \alpha} = \frac{\partial u(t)}{\partial \alpha} H_d.$$

### 3. The $\texttt{Propagation}$ Module

The $\texttt{Propagation}$ module computes the derivative of the unitary operator generated by the time-dependent Hamiltonian by using quantum optimal control methods like GRAPE and GOAT. These methods work for specific pulse ansatz, for e.g. GRAPE require a piecewise constant (PWC) ansatz. These methods provide ways to propagate the derivative of the Hamiltonian to compute derivative of the unitary operator.